# Stage 3c: full neural sweeps

This notebook runs every neural model × fold × seed for the station-blocked benchmark, then evaluates the results against the Stage 2 baselines.

- It runs in Jupyter on Windows, macOS or Linux, or on Google Colab. A GPU is used automatically if one is available.
- On a CPU, each run takes about 30–60 s, so the full sweep of 100 runs takes about 1–3 hours.

**Stopping and resuming.** You can stop at any time (interrupt the kernel, close the laptop, lose the connection) and run the same cells again:
- **Training:** neural runs save a checkpoint after every epoch, and a stopped run continues from its last completed epoch with exactly the same result.
- **Finished runs** are skipped.
- **Baselines** are saved fold by fold.
- **Panel and leakage test:** the clean panel and the leakage test are not redone once they exist.

**Before running:** see `data/README.md` for the two raw data files.

In [ ]:
# 1. Get the code (skip if you already cloned the repository and opened this notebook from it)
import os, subprocess, sys
REPO = 'cookfarm-soilwater-reconstruction'
if not os.path.exists('scripts/09_neural.py'):
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', f'https://github.com/saurabh-chy/{REPO}.git'], check=True)
    os.chdir(REPO)
print(os.getcwd())

In [ ]:
# 2. Install dependencies (works in Jupyter on Windows, macOS or Linux, and on Colab)
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
if r.returncode != 0:  # pinned versions unavailable for this Python: fall back to current releases
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'scipy', 'scikit-learn',
                    'lightgbm', 'pyarrow', 'pyshp', 'rasterio', 'shapely'], check=True)
try:
    import torch
except ImportError:  # CPU build of PyTorch; for an NVIDIA GPU see https://pytorch.org/get-started/locally/
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], check=True)
    import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())

In [ ]:
# 3. Raw data. If the downloads below fail, copy your two files into the repository's data/raw/ folder
#    and rename them CAF_Sensor_Dataset_2.zip and daymet_caf_2007_2016.csv, then rerun this cell
import os, zipfile, urllib.request
os.makedirs('data/raw', exist_ok=True)
CAF_ZIP = 'data/raw/CAF_Sensor_Dataset_2.zip'
if not os.path.exists('data/raw/CAF'):
    if not os.path.exists(CAF_ZIP):
        urllib.request.urlretrieve('https://ndownloader.figshare.com/files/44335805', CAF_ZIP)
    zipfile.ZipFile(CAF_ZIP).extractall('data/raw/CAF')
DAYMET = 'data/raw/daymet_caf_2007_2016.csv'
if not os.path.exists(DAYMET):
    url = ('https://daymet.ornl.gov/single-pixel/api/data?lat=46.7815&lon=-117.082'
           '&vars=prcp,tmax,tmin,srad,vp,swe,dayl&start=2007-01-01&end=2016-12-31')
    urllib.request.urlretrieve(url, DAYMET)  # if this fails, download it in a browser (see data/README.md)
import hashlib
for f in [CAF_ZIP, DAYMET]:
    if os.path.exists(f):
        print(hashlib.sha256(open(f, 'rb').read()).hexdigest(), f)  # compare with data/README.md
    else:
        print('not present (already unzipped):', f)

In [ ]:
# 4. Build the clean panel, confirm the frozen folds, and run the baselines (needed for evaluation; ~6 min)
import filecmp
def run(*args):
    r = subprocess.run([sys.executable, '-I', *args], capture_output=True, text=True)
    print(r.stdout[-1500:]); assert r.returncode == 0, r.stderr[-3000:]
if not os.path.exists('data/processed/panel.parquet'):
    run('scripts/02_build_panel.py', 'data/raw/CAF', 'data/raw/daymet_caf_2007_2016.csv', 'data/processed')
else:
    print('clean panel already built: skipping')
run('scripts/03_make_folds.py', 'data/processed')
print('folds match config/folds.csv:', filecmp.cmp('data/processed/folds_frozen_format.csv', 'config/folds.csv', shallow=False))
os.environ['FOLDS'] = 'config/folds.csv'
if not os.path.exists('results/predictions.parquet'):
    run('scripts/04_baselines.py', 'data/processed', 'results', '--leak-test')
    run('scripts/04_baselines.py', 'data/processed', 'results')

In [ ]:
# 5. Sweep: models x folds x seeds.
#    Finished runs are skipped; a run that was stopped part-way resumes from its last epoch checkpoint.
import itertools, glob, time
MODELS = ['gcn', 'attn', 'srn', 'srn_unshared']
FOLDS = range(5)
SEEDS = range(5)
EXTRA = []        # e.g. ['--kappa-mode', 'free'] for the scale-decay variants
threads = str(os.cpu_count() or 2)
jobs = list(itertools.product(SEEDS, FOLDS, MODELS))
t0 = time.time()
for i, (seed, fold, model) in enumerate(jobs, 1):
    tag = f'{model}_f{fold}_s{seed}'
    if os.path.exists(f'results/neural/{tag}.parquet'):
        continue
    resuming = os.path.exists(f'results/neural/checkpoints/{tag}.pt')
    print(f'[{i}/{len(jobs)}] {tag}' + (' (resuming)' if resuming else ''), end=' ... ', flush=True)
    r = subprocess.run([sys.executable, '-I', 'scripts/09_neural.py', '--model', model, '--fold', str(fold),
                        '--seed', str(seed), '--threads', threads] + EXTRA, capture_output=True, text=True)
    print('ok' if r.returncode == 0 else 'FAILED\n' + r.stderr[-800:], f'| {time.time()-t0:.0f}s', flush=True)
done = len(glob.glob('results/neural/*_f[0-4]_s[0-9].parquet'))
print(f'{done} of {len(jobs)} runs finished')

In [ ]:
# 6. Leakage test (fold 0): corrupting test-station readings must not change predictions (expect 0.0)
import pandas as pd
for m in MODELS:
    for extra in [[], ['--leak-test']]:
        tag = f'{m}_f0_s0' + ('_leak' if extra else '')
        if not os.path.exists(f'results/neural_leak/{tag}.parquet'):
            subprocess.run([sys.executable, '-I', 'scripts/09_neural.py', '--model', m, '--fold', '0', '--seed', '0',
                            '--out', 'results/neural_leak'] + extra, capture_output=True)
    a = pd.read_parquet(f'results/neural_leak/{m}_f0_s0.parquet'); b = pd.read_parquet(f'results/neural_leak/{m}_f0_s0_leak.parquet')
    x = a.merge(b, on=['date', 'station', 'depth'], suffixes=('', '_l'))
    print(m, 'max |pred diff| =', float((x.pred - x.pred_l).abs().max()))

In [ ]:
# 7. Evaluate all seeds against the baselines (both targets)
run('scripts/10_evaluate_neural.py', '--seeds', '0', '1', '2', '3', '4')

In [ ]:
# 8. Bundle the results to upload to the chat (per-run .parquet files are large and left out)
import zipfile, glob
with zipfile.ZipFile('neural_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob('results/neural/*.json') + glob.glob('results/neural_eval/*') + glob.glob('results/neural_leak/*.json'):
        z.write(f)
print('Created', os.path.abspath('neural_results.zip'))